#  Web Scraping and Text Analysis — Books to Scrape

**Dataset:** the 50 Books to Scrape product pages previously collected with BeautifulSoup.  
**Analysis:** VADER sentiment + TF-IDF topic relevance + concise extractive summaries.

This revision reuses the collected dataset rather than making another 50 network requests. The optional scraping cell is retained for reproducibility. The topic is framed as **book discovery and reader appeal**, which fits the actual content of product descriptions better than the previous overly narrow query.

## 1. Why BeautifulSoup rather than Scrapy?

BeautifulSoup + Requests is selected because this is a small, bounded 50-page project and the work is easy to demonstrate in a notebook. Scrapy is a better fit for large, concurrent, repeatable crawls because it provides request scheduling, retries, pipelines and concurrency as a framework. BeautifulSoup is simpler for a one-off extraction task.

The notebook does not claim a measured Scrapy-vs-BeautifulSoup benchmark. Its optional benchmark measures BeautifulSoup parsing throughput on downloaded HTML only; a fair framework comparison would require equivalent crawler configurations and network conditions.

## 2. Install and import dependencies

In [ ]:
%pip install -q requests beautifulsoup4 pandas numpy scikit-learn vaderSentiment matplotlib

In [ ]:
import re, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from IPython.display import display

## 3. Load the existing 50-page dataset and repair text encoding

Place `scraped_books_raw.csv` in the same folder as this notebook. This cell loads the collected pages, removes duplicate URLs, and repairs common UTF-8/Windows-1252 mojibake (for example, `Â£` to `£`).

In [ ]:
INPUT_FILE = "scraped_books_raw.csv"
df = pd.read_csv(INPUT_FILE, encoding="utf-8")

def repair_text(value):
    if pd.isna(value):
        return ""
    text = str(value)
    for _ in range(2):
        if any(marker in text for marker in ["Ã", "Â", "â€", "â€™", "â€œ", "â€”", "â"]):
            try:
                text = text.encode("cp1252").decode("utf-8")
            except (UnicodeEncodeError, UnicodeDecodeError):
                break
    text = text.replace("\\xa0", " ").replace("\\u200b", " ")
    return re.sub(r"\\s+", " ", text).strip()

for col in ["title", "description", "category", "price"]:
    if col in df.columns:
        df[col] = df[col].apply(repair_text)

df["title"] = df["title"].fillna("")
df["description"] = df["description"].fillna("")
df["category"] = df["category"].fillna("Uncategorized")
df["text"] = (df["title"] + ". " + df["description"]).str.strip()
df = df.drop_duplicates(subset="url").reset_index(drop=True)

print("Unique product pages:", len(df))
print("Missing descriptions:", (df["description"].str.len() == 0).sum())
display(df[["title", "category", "price", "rating", "url"]].head())

## 4. Optional: original BeautifulSoup collection logic

The 50 records already exist, so **do not run this cell unless you want to repeat the crawl**. The target is a public practice website. Respect the site's terms and crawl policies; use delays and stop if access is denied. User-Agent rotation is not a way to bypass blocking or access controls.

In [ ]:
# OPTIONAL RE-CRAWL — disabled by default
RUN_LIVE_CRAWL = False

if RUN_LIVE_CRAWL:
    BASE = "https://books.toscrape.com/"
    USER_AGENTS = [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_6) AppleWebKit/605.1.15 Version/17.0 Safari/605.1.15",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/123.0 Safari/537.36",
    ]
    session = requests.Session()

    def fetch_soup(url):
        response = session.get(url, headers={"User-Agent": random.choice(USER_AGENTS)}, timeout=20)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")

    product_urls, listing_url = [], BASE
    while listing_url and len(product_urls) < 50:
        soup = fetch_soup(listing_url)
        product_urls.extend(urljoin(listing_url, a["href"]) for a in soup.select("article.product_pod h3 a"))
        nxt = soup.select_one("li.next a")
        listing_url = urljoin(listing_url, nxt["href"]) if nxt else None
        time.sleep(random.uniform(0.6, 1.2))

    rows = []
    for url in list(dict.fromkeys(product_urls))[:50]:
        soup = fetch_soup(url)
        title = soup.select_one("div.product_main h1")
        price = soup.select_one("div.product_main p.price_color")
        rating = soup.select_one("div.product_main p.star-rating")
        desc = soup.select_one("#product_description + p")
        category = soup.select_one("ul.breadcrumb li:nth-of-type(3) a")
        rows.append({
            "url": url,
            "title": title.get_text(" ", strip=True) if title else "",
            "price": price.get_text(" ", strip=True) if price else "",
            "rating": rating.get("class", ["", ""])[-1] if rating else "",
            "category": category.get_text(" ", strip=True) if category else "",
            "description": desc.get_text(" ", strip=True) if desc else ""
        })
        time.sleep(random.uniform(0.6, 1.2))
    df = pd.DataFrame(rows)
    df.to_csv("scraped_books_raw.csv", index=False, encoding="utf-8-sig")
    print("Collected:", len(df))

## 5. Algorithm 1 — VADER sentiment

VADER produces a compound score from -1 (strongly negative) to +1 (strongly positive). The conventional thresholds are:
- compound ≥ 0.05: Positive
- compound ≤ -0.05: Negative
- otherwise: Neutral

Interpretation caution: these are promotional book descriptions, so sentiment describes the **language of the description**, not an objective review of the book's quality.

In [ ]:
vader = SentimentIntensityAnalyzer()
df["sentiment_compound"] = df["text"].apply(
    lambda t: vader.polarity_scores(t)["compound"]
)
df["sentiment_label"] = df["sentiment_compound"].apply(
    lambda x: "Positive" if x >= 0.05 else ("Negative" if x <= -0.05 else "Neutral")
)
display(df["sentiment_label"].value_counts().rename_axis("Sentiment").to_frame("Pages"))

## 6. Algorithm 2 — TF-IDF relevance, 

The previous query was too narrow and produced extremely small raw cosine similarities. This version uses a broader vocabulary aligned with the actual dataset: book discovery, reader appeal, genre, narrative, themes, authors, and audience.

Cosine similarity remains the underlying TF-IDF measure. Because raw similarity values are small and difficult to interpret, scores are converted to **within-dataset percentile relevance (0–100)**. A score of 90 means the page is more relevant than roughly 90% of the 50 pages in this dataset. It does **not** mean 90% absolute relevance.

This ranking-based transformation improves interpretability but makes the score relative to this particular sample.

In [ ]:
TOPIC = (
    "book discovery reader appeal genre story narrative characters themes author "
    "writing style audience literary fiction nonfiction poetry history romance "
    "mystery fantasy adventure biography children books"
)
vectorizer = TfidfVectorizer(
    stop_words="english", ngram_range=(1, 2),
    sublinear_tf=True, min_df=1, max_df=0.95
)
X = vectorizer.fit_transform(df["text"])
topic_vector = vectorizer.transform([TOPIC])
df["topic_similarity_raw"] = cosine_similarity(X, topic_vector).ravel()

# Percentile relevance, 0–100, calculated across this 50-page sample.
df["relevance_score_100"] = (
    df["topic_similarity_raw"].rank(pct=True, method="average") * 100
).round(2)

## 7.  extractive summaries

The notebook scores each sentence using the average TF-IDF weight of its terms, selects up to two informative sentences, preserves their original order, and limits output to 60 words. This produces extractive summaries: wording comes from the source description rather than a generative model.

In [ ]:
def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\\s+(?=[A-Z0-9“"])', str(text))
    return [p.strip() for p in parts if len(p.strip()) > 25]

def make_summary(text):
    sents = split_sentences(text)
    if not sents:
        return str(text)[:300]
    matrix = vectorizer.transform(sents)
    sums = np.asarray(matrix.sum(axis=1)).ravel()
    counts = np.maximum(np.asarray((matrix != 0).sum(axis=1)).ravel(), 1)
    scores = sums / counts
    ranked = list(np.argsort(scores)[::-1])
    chosen = []
    seen = set()
    for idx in ranked:
        normalized = re.sub(r"\\W+", " ", sents[idx].lower()).strip()
        if normalized and normalized not in seen:
            chosen.append(int(idx))
            seen.add(normalized)
        if len(chosen) == 2:
            break
    chosen.sort()
    summary = " ".join(sents[i] for i in chosen)
    words = summary.split()
    if len(words) > 60:
        summary = " ".join(words[:60]).rstrip(" ,;:") + "…"
    return summary

df["summary"] = df["text"].apply(make_summary)

## 8. Directional importance score

The score combines relative topic relevance and VADER sentiment:

`Directional Importance = Relevance Percentile × VADER Compound`

Range is approximately -100 to +100. The separate direction label uses explicit rules:
- **Low relevance:** relevance percentile below 20, regardless of sentiment.
- **Positive:** relevance ≥ 20 and VADER ≥ 0.05.
- **Negative:** relevance ≥ 20 and VADER ≤ -0.05.
- **Neutral:** relevance ≥ 20 and sentiment between -0.05 and +0.05.

The low-relevance cutoff is an analytical choice for this dataset, not a universal statistical standard. Keeping the numerical score and categorical direction separate avoids hiding sentiment direction just because a score is small.

In [ ]:
df["directional_importance"] = (
    df["relevance_score_100"] * df["sentiment_compound"]
).round(2)

df["importance_direction"] = np.where(
    df["relevance_score_100"] < 20, "Low relevance",
    np.where(df["sentiment_compound"] >= 0.05, "Positive",
             np.where(df["sentiment_compound"] <= -0.05, "Negative", "Neutral"))
)

## 9. Final results table — all 50 pages

In [ ]:
results = df[[
    "title", "category", "rating", "summary",
    "sentiment_compound", "sentiment_label",
    "relevance_score_100", "directional_importance",
    "importance_direction", "url"
]].sort_values("directional_importance", ascending=False).reset_index(drop=True)

pd.set_option("display.max_colwidth", 120)
display(results)

## 10. Visual presentation and interpretation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["importance_direction"].value_counts().reindex(
    ["Positive", "Negative", "Neutral", "Low relevance"], fill_value=0
).plot(kind="bar", ax=ax)
ax.set_title("Directional importance classification (50 book pages)")
ax.set_xlabel("Classification")
ax.set_ylabel("Number of pages")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["relevance_score_100"], bins=10)
ax.set_title("TF-IDF relevance percentile distribution")
ax.set_xlabel("Relative relevance score (0–100)")
ax.set_ylabel("Number of pages")
plt.tight_layout()
plt.show()

print("Mean VADER compound:", round(df["sentiment_compound"].mean(), 3))
print("Mean relevance percentile:", round(df["relevance_score_100"].mean(), 2))
print("Direction counts:")
display(df["importance_direction"].value_counts().to_frame("Pages"))

## 11. Export deliverables

In [ ]:
df.to_csv("scraped_books_raw_cleaned.csv", index=False, encoding="utf-8-sig")
results.to_csv("scraped_text_analysis_results.csv", index=False, encoding="utf-8-sig")
print("Exported cleaned raw data and the  50-page results table.")

## 12. Limitations and responsible interpretation

1. VADER measures sentiment expressed in marketing copy, not independent book quality.
2. TF-IDF is lexical: it rewards overlap in vocabulary and does not understand semantic meaning like a transformer model.
3. Percentile relevance is relative to these 50 pages; it should not be compared directly across unrelated datasets.
4. The 20th-percentile low-relevance cutoff is a transparent heuristic.
5. Summaries are extractive and may not capture every nuance.
6. The notebook reuses the 50 previously collected pages; live re-scraping is optional.
7. User-Agent rotation is not intended to evade access restrictions. Crawl politely and respect site rules.